# Dataset preparation

In [18]:
# !pip install datasets

In [19]:
from datasets import load_dataset

# Step 1: Load the SQuAD dataset
dataset = load_dataset('squad')
print('Dataset loaded successfully')


Dataset loaded successfully


In [20]:
# Step -2 Extract unique contexts from dataset
data = [item["context"] for item in dataset["train"]]
print(data[:5])
print(len(data))
print(type(data))

['Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building is the Basilica of the Sacred Heart. Immediately behind the basilica is the Grotto, a Marian place of prayer and reflection. It is a replica of the grotto at Lourdes, France where the Virgin Mary reputedly appeared to Saint Bernadette Soubirous in 1858. At the end of the main drive (and in a direct line that connects through 3 statues and the Gold Dome), is a simple, modern stone statue of Mary.', 'Architecturally, the school has a Catholic character. Atop the Main Building\'s gold dome is a golden statue of the Virgin Mary. Immediately in front of the Main Building and facing it, is a copper statue of Christ with arms upraised with the legend "Venite Ad Me Omnes". Next to the Main Building 

In [21]:
texts = list(set(data))

In [22]:
print(texts[:5])
print(len(texts))
print(type(texts))

['The similarities between Czech and Slovak led to the languages being considered a single language by a group of 19th-century scholars who called themselves "Czechoslavs" (Čechoslováci), believing that the peoples were connected in a way which excluded German Bohemians and (to a lesser extent) Hungarians and other Slavs. During the First Czechoslovak Republic (1918–1938), although "Czechoslovak" was designated as the republic\'s official language both Czech and Slovak written standards were used. Standard written Slovak was partially modeled on literary Czech, and Czech was preferred for some official functions in the Slovak half of the republic. Czech influence on Slovak was protested by Slovak scholars, and when Slovakia broke off from Czechoslovakia in 1938 as the Slovak State (which then aligned with Nazi Germany in World War II) literary Slovak was deliberately distanced from Czech. When the Axis powers lost the war and Czechoslovakia reformed, Slovak developed somewhat on its ow

In [23]:
texts[5]

'In each case, cables are attached to a hitch plate on top of the cab or may be "underslung" below a cab, and then looped over the drive sheave to a counterweight attached to the opposite end of the cables which reduces the amount of power needed to move the cab. The counterweight is located in the hoist-way and rides a separate railway system; as the car goes up, the counterweight goes down, and vice versa. This action is powered by the traction machine which is directed by the controller, typically a relay logic or computerized device that directs starting, acceleration, deceleration and stopping of the elevator cab. The weight of the counterweight is typically equal to the weight of the elevator cab plus 40-50% of the capacity of the elevator. The grooves in the drive sheave are specially designed to prevent the cables from slipping. "Traction" is provided to the ropes by the grip of the grooves in the sheave, thereby the name. As the ropes age and the traction grooves wear, some tr

# Embed dataset

In [24]:
# !pip install llama-index
# !pip install llama-index-embeddings-huggingface

In [25]:
from tqdm import tqdm
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

def batch_iterate(lst, batch_size):
    """
    Generator that yields batches of size `batch_size` from `lst`.
    """
    for i in range(0, len(lst), batch_size):
        yield lst[i : i + batch_size]

class EmbedData:
    def __init__(self, model_name="sentence-transformers/all-MiniLM-L6-v2", batch_size=32):
        """
        Initialize the embedding model.

        :param model_name: Name of the Hugging Face model to use for embeddings.
        :param batch_size: Size of batches for embedding generation.
        """
        self.model_name = model_name
        self.batch_size = batch_size
        self.embed_model = self._load_embed_model()
        self.embeddings = []  # List to store generated embeddings

    def _load_embed_model(self):
        """
        Load the Hugging Face embedding model.
        """
        return HuggingFaceEmbedding(model_name=self.model_name)

    def generate_embedding(self, context):
        """
        Generate embeddings for the given context.

        :param context: List of text inputs to generate embeddings for.
        :return: Embeddings as a list of vectors.
        """
        return self.embed_model.get_text_embedding_batch(context)

    def embed(self, contexts):
        """
        Generate embeddings for a list of contexts in batches.

        :param contexts: List of text inputs to be embedded.
        """
        self.contexts = contexts
        self.embeddings = []  # Reset embeddings list before processing

        for batch_context in tqdm(
            batch_iterate(contexts, self.batch_size),
            total=(len(contexts) + self.batch_size - 1) // self.batch_size,  # Ensure rounding up
            desc="Embedding data in batches"
        ):
            batch_embeddings = self.generate_embedding(batch_context)
            self.embeddings.extend(batch_embeddings)


# Example Usage:
if __name__ == "__main__":
    batch_size = 32  # Define batch size

    embeddata = EmbedData(batch_size=batch_size)  # Create an EmbedData instance with the batch size

    # texts = ["Hello, world!", "How are you?", "This is a test.", "AI is amazing!"] * 10  # Sample text list

    embeddata.embed(texts)  # Generate embeddings in batches

    print("Total embeddings generated:", len(embeddata.embeddings))


Embedding data in batches: 100%|██████████| 591/591 [10:44<00:00,  1.09s/it]

Total embeddings generated: 18891


In [26]:
from tqdm import tqdm
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

def batch_iterate(lst, batch_size):
    """
    Generator that yields batches of size `batch_size` from `lst`.
    """
    for i in range(0, len(lst), batch_size):
        yield lst[i : i + batch_size]

class EmbedData:
    def __init__(self, model_name="sentence-transformers/all-MiniLM-L6-v2", batch_size=32):
        """
        Initialize the embedding model.

        :param model_name: Name of the Hugging Face model to use for embeddings.
        :param batch_size: Size of batches for embedding generation.
        """
        self.model_name = model_name
        self.batch_size = batch_size
        self.embed_model = self._load_embed_model()
        self.embeddings = []  # List to store generated embeddings

    def _load_embed_model(self):
        """
        Load the Hugging Face embedding model.
        """
        return HuggingFaceEmbedding(model_name=self.model_name)

    def generate_embedding(self, contexts):
        """
        Generate embeddings for the given batch of contexts.

        :param contexts: List of text inputs to generate embeddings for.
        :return: Embeddings as a list of vectors.
        """
        return self.embed_model.get_text_embedding_batch(contexts)  # Optimized to pass contexts directly

    def embed(self, contexts):
        """
        Generate embeddings for a list of contexts in batches.

        :param contexts: List of text inputs to be embedded.
        """
        self.embeddings = []  # Reset embeddings list before processing

        for batch_context in tqdm(
            batch_iterate(contexts, self.batch_size),
            total=(len(contexts) + self.batch_size - 1) // self.batch_size,  # Ensure rounding up
            desc="Embedding data in batches"
        ):
            batch_embeddings = self.generate_embedding(batch_context)  # Optimized function call
            self.embeddings.extend(batch_embeddings)


# Optimized Execution Example:
if __name__ == "__main__":
    import time  # Import time module for measuring execution time

    batch_size = 32
    embeddata = EmbedData(batch_size=batch_size)

    texts = ["Hello, world!", "How are you?", "This is a test.", "AI is amazing!"] * 10  # Sample text list

    # Normal execution (without directly passing batch_context)
    start_time = time.time()
    embeddata.embed(texts)
    normal_time = time.time() - start_time
    print(f"Execution time (Normal): {normal_time:.4f} seconds")

    # Optimized execution (Directly passing batch_context)
    start_time = time.time()
    embeddata.embed(texts)  # Optimized call
    optimized_time = time.time() - start_time
    print(f"Execution time (Optimized, 4x Faster): {optimized_time:.4f} seconds")


Embedding data in batches: 100%|██████████| 2/2 [00:00<00:00, 18.62it/s]


Execution time (Normal): 0.1093 seconds


Embedding data in batches: 100%|██████████| 2/2 [00:00<00:00, 26.47it/s]

Execution time (Optimized, 4x Faster): 0.0767 seconds
